# S&P 500 주가 데이터 수집 (Polars)
yfinance로 S&P 500 데이터를 수집하고 Polars DataFrame으로 처리합니다.

In [1]:
# !pip install yfinance polars pandas requests lxml pytz pyarrow -q

In [5]:
import yfinance as yf
import polars as pl
import pandas as pd
import requests
from io import StringIO

def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    df = pd.read_html(StringIO(response.text))[0]
    # df = pl.from_pandas(pdf)
    # tickers = df['Symbol'].str.replace('.', '-', literal=True).to_list()
    tickers = df['Symbol'].str.replace('.', '-').to_list()
    return tickers, df

tickers, sp500_df = get_sp500_tickers()
print(f'S&P 500 종목 수: {len(tickers)}')
# sp500_df.select(['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry']).head(10)

S&P 500 종목 수: 503


In [6]:
sp500_df

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [7]:
info = yf.Ticker('AAPL').info
market_cap = info.get('marketCap')
pl.DataFrame({
    '항목': ['종목명', '섹터', '시가총액', '현재가', '52주 최고', '52주 최저'],
    '값': [
        str(info.get('longName')),
        str(info.get('sector')),
        f'${market_cap:,}' if market_cap else 'N/A',
        str(info.get('currentPrice')),
        str(info.get('fiftyTwoWeekHigh')),
        str(info.get('fiftyTwoWeekLow')),
    ]
})

항목,값
str,str
"""종목명""","""Apple Inc."""
"""섹터""","""Technology"""
"""시가총액""","""$4,057,771,802,624"""
"""현재가""","""276.65"""
"""52주 최고""","""288.62"""
"""52주 최저""","""193.25"""


In [8]:
df_aapl = pl.from_pandas(
    yf.download('AAPL', start='2023-01-01', end='2024-12-31', progress=False).reset_index()
)
print(f'데이터 크기: {df_aapl.shape}')
df_aapl.tail()

ImportError: pyarrow is required for converting a pandas dataframe to Polars, unless each of its columns is a simple numpy-backed one (e.g. 'int64', 'bool', 'float32' - not 'Int64')

In [ ]:
top10 = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'META', 'GOOGL', 'GOOG', 'BRK-B', 'LLY', 'AVGO']
raw = yf.download(top10, start='2024-01-01', end='2024-12-31', progress=False)
close_prices = pl.from_pandas(raw['Close'].reset_index())
print(f'데이터 크기: {close_prices.shape}')
close_prices.tail()

In [ ]:
df_all_pd = yf.download(tickers, period='1y', interval='1d', auto_adjust=True, threads=True, progress=True)
close_all = pl.from_pandas(df_all_pd['Close'].reset_index())
print(f'다운로드 완료! 데이터 크기: {close_all.shape}')
close_all.tail()

In [ ]:
total_rows = close_all.height
date_col = 'Date'
missing_ratio = (
    close_all.select(pl.all().exclude(date_col))
    .select((pl.all().null_count() / total_rows * 100).name.suffix('_null_pct'))
)
valid_cols = [
    col.replace('_null_pct', '')
    for col, val in zip(missing_ratio.columns, missing_ratio.row(0))
    if val < 20
]
close_clean = close_all.select([date_col] + valid_cols).fill_null(strategy='forward')
print(f'유효 종목 수: {len(valid_cols)}, 최종 데이터 크기: {close_clean.shape}')
close_clean.tail()

In [ ]:
close_clean.write_csv('sp500_close_prices.csv')
sp500_df.write_csv('sp500_info.csv')
print('저장 완료: sp500_close_prices.csv, sp500_info.csv')

## 직전 거래일 등락률 계산 (한국 오전 7시 실행 기준)

| 시각 | 설명 |
|------|------|
| KST 07:00 | 한국 실행 시각 |
| EST 17:00 / EDT 18:00 | 미국 동부 시각 (장 마감 16:00 ET 이후) |

오전 7시 실행 시 T-1 종가는 항상 확정 상태이므로 T-1 vs T-2 등락률 계산 가능

In [ ]:
from datetime import datetime
import pytz

kst = pytz.timezone('Asia/Seoul')
now_kst = datetime.now(kst)
print(f'실행 시각 (KST): {now_kst.strftime("%Y-%m-%d %H:%M:%S")}')

# 최근 5거래일 다운로드 (주말/공휴일 고려)
df_recent = pl.from_pandas(
    yf.download(tickers, period='5d', interval='1d', auto_adjust=True, progress=False)
    ['Close'].reset_index()
)

last2 = df_recent.tail(2)
dates = last2['Date'].to_list()
prev_date, curr_date = dates[0], dates[1]
print(f'T-2 (전전 거래일): {prev_date}')
print(f'T-1 (직전 거래일): {curr_date}')

In [ ]:
returns_df = (
    last2
    .unpivot(index='Date', variable_name='ticker', value_name='close')
    .sort(['ticker', 'Date'])
    .with_columns(
        pl.col('close').shift(1).over('ticker').alias('prev_close')
    )
    .filter(pl.col('Date') == curr_date)
    .drop_nulls('prev_close')
    .with_columns(
        ((pl.col('close') - pl.col('prev_close')) / pl.col('prev_close') * 100)
        .round(2)
        .alias('return_pct')
    )
    .select(['ticker', 'prev_close', 'close', 'return_pct'])
    .sort('return_pct', descending=True)
)

print(f'계산 종목 수: {returns_df.height}')
print('\n상승률 TOP 10')
print(returns_df.head(10))
print('\n하락률 TOP 10')
print(returns_df.tail(10))

In [ ]:
# 섹터별 평균 등락률
sector_df = (
    sp500_df
    .select(['Symbol', 'GICS Sector'])
    .rename({'Symbol': 'ticker'})
    .with_columns(pl.col('ticker').str.replace('.', '-', literal=True))
)

sector_returns = (
    returns_df
    .join(sector_df, on='ticker', how='left')
    .group_by('GICS Sector')
    .agg(
        pl.col('return_pct').mean().round(2).alias('avg_return_pct'),
        pl.col('ticker').count().alias('종목수')
    )
    .sort('avg_return_pct', descending=True)
)
sector_returns